# Benchmark retrieval embedding cho hệ thống Legal RAG

Notebook đọc kết quả của một **phiên bản benchmark** đã đóng băng (`labor_rag/evaluation/benchmarks/<dataset_id>/`, xem `docs/BENCHMARK.md`): bộ câu hỏi cố định trên đúng một phiên bản dữ liệu (`lao_dong_v5`: ontology v4, chunk schema v5). Ba mô hình embedding được so trên cùng dữ liệu đó.


## 1. Bài toán và phạm vi benchmark

### Bài toán cần giải quyết

Trong hệ thống Legal RAG, người dùng đặt một câu hỏi pháp luật bằng tiếng Việt. Hệ thống cần tìm ra các Điều luật liên quan nhất để cung cấp bằng chứng cho câu trả lời. Embedding model chuyển câu hỏi và nội dung Điều luật thành vector; Qdrant dùng các vector đó để tìm các Điều luật gần nghĩa nhất.

Câu hỏi nghiên cứu của benchmark là: **trên cùng một tập Điều luật và cùng một bộ câu hỏi, mô hình embedding nào tìm được Điều luật đúng tốt nhất với thời gian truy vấn phù hợp?**

### Phạm vi của thí nghiệm

Benchmark này chỉ đo chất lượng vector retrieval để cô lập ảnh hưởng của embedding model:

```text
Question → Embedding model → Qdrant vector search → Top-k Điều luật → Metrics
```

Neo4j và LLM vẫn thuộc pipeline Legal RAG tổng thể, nhưng không tham gia metric chính trong notebook này. Neo4j sẽ phục vụ mở rộng quan hệ pháp lý; LLM sẽ sinh câu trả lời và citation ở bước đánh giá hybrid riêng.

### Pipeline dữ liệu

```text
ABox TTL (ontology v4) → data/processed/lao_dong_v5
        ↓
Projection JSONL
        ↓
chunks.jsonl gồm các Điều luật
        ↓
Mỗi embedding model tạo một collection Qdrant riêng
        ↓
Truy vấn bộ câu hỏi đã đóng băng
        ↓
HitRate, Recall, MultiHit, MRR, nDCG (kèm CI 95%) và latency
```

TTL/ABox là nguồn dữ liệu ngữ nghĩa chuẩn. `chunks.jsonl` là projection phục vụ retrieval. Qdrant lưu vector của từng model, còn notebook kiểm tra model nào đưa Điều luật đúng lên vị trí cao hơn và với tốc độ nào.

## 2. Dữ liệu benchmark

Chunk lấy từ `data/processed/<dataset_id>/chunks.jsonl`; Điều dài được cắt thành nhiều phần cùng `mapc`, và kết quả được gom theo `mapc` trước khi chấm. Bộ câu hỏi là file đã đóng băng trong `benchmarks/<dataset_id>/`, dùng chung cho cả ba model.


In [ ]:
import json
from collections import Counter
from pathlib import Path
import pandas as pd
from IPython.display import display

BENCHMARK = 'lao_dong_v5'   # phiên bản dữ liệu
TESTSET = 'v1'              # bộ câu hỏi đã đóng băng trong benchmarks/<BENCHMARK>/

overview_root = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / 'labor_rag' / 'evaluation' / 'benchmarks' / BENCHMARK / f'{TESTSET}.jsonl').exists()
)
overview_questions = [
    json.loads(line)
    for line in (overview_root / 'labor_rag' / 'evaluation' / 'benchmarks' / BENCHMARK / f'{TESTSET}.jsonl').read_text(encoding='utf-8').splitlines()
    if line.strip()
]
question_category_counts = (
    pd.Series(Counter(str(question.get('category', 'uncategorized')) for question in overview_questions), name='question_count')
    .rename_axis('category')
    .reset_index()
    .sort_values(['question_count', 'category'], ascending=[False, True])
)
display(question_category_counts)
assert int(question_category_counts['question_count'].sum()) == len(overview_questions)
print(f'Tổng số câu hỏi theo nhóm: {int(question_category_counts["question_count"].sum())}')

### Bộ câu hỏi vàng và nhãn liên quan

Bộ test gồm các câu hỏi thủ công về quyền, nghĩa vụ, điều kiện, trách nhiệm, chế độ, cơ quan có thẩm quyền, điều cấm, định nghĩa, lao động đặc thù và hiệu lực. Mỗi câu hỏi liên kết với một hoặc nhiều `mapc` trong `chunks.jsonl`.

| Điểm relevance | Ý nghĩa |
|---:|---|
| `3` | Điều luật trực tiếp trả lời câu hỏi |
| `2` | Chứa một phần quan trọng của câu trả lời |
| `1` | Chỉ cho ngữ cảnh; **không** tính là đúng trong HitRate/Recall/MRR (`--min-grade 2`), vẫn được điểm trong nDCG |

Gold label được gán thủ công, không dùng LLM để tự tạo nhãn. Vì vậy kết quả phản ánh khả năng truy hồi so với bộ đáp án chuẩn đã định nghĩa trước.

In [ ]:
from pathlib import Path
import sys

candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next(
    path for path in candidates
    if (path / 'labor_rag' / 'evaluation' / 'embedding_benchmark.py').exists()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from labor_rag.evaluation.benchmark_version import resolve
from labor_rag.evaluation.embedding_benchmark import DEFAULT_RESULTS, METRICS, MODELS, read_jsonl, with_collection_prefix

benchmark = resolve(BENCHMARK, TESTSET)   # dừng nếu dữ liệu đã đổi so với lúc đóng băng
for warning in benchmark['warnings']:
    print('WARN', warning)
CHUNKS_PATH = benchmark['chunks_path']
QUESTIONS_PATH = benchmark['questions_path']
MODELS = with_collection_prefix(MODELS, benchmark['collection_prefix'])
RUNS_DIR = DEFAULT_RESULTS / BENCHMARK / TESTSET
runs = sorted(path.parent for path in RUNS_DIR.glob('*/summary.csv'))
RESULTS_DIR = runs[-1] if runs else RUNS_DIR / 'chua_chay'
SUMMARY_PATH = RESULTS_DIR / 'summary.csv'
METRICS_PATH = RESULTS_DIR / 'metrics.csv'
BREAKDOWN_PATH = RESULTS_DIR / 'breakdown.csv'
PER_QUERY_PATH = RESULTS_DIR / 'per_query_results.jsonl'
print(f'Workspace: {ROOT}')
print(f'Benchmark: {BENCHMARK}/{TESTSET}')
print(f'Chunks: {CHUNKS_PATH}')
print(f'Results (lần chạy mới nhất): {RESULTS_DIR}')


## 3. Mô hình và cấu hình benchmark

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

model_table = pd.DataFrame(MODELS).rename(
    columns={'key': 'key', 'model': 'Hugging Face model', 'collection': 'Qdrant collection'}
)
display(model_table)

chunks = read_jsonl(CHUNKS_PATH)
questions = read_jsonl(QUESTIONS_PATH)
chunk_mapcs = {str(chunk['mapc']) for chunk in chunks}
gold_mapcs = {str(mapc) for question in questions for mapc in question['relevance']}
missing_gold = sorted(gold_mapcs - chunk_mapcs)
validation = pd.DataFrame([{
    'chunk_count': len(chunks),
    'unique_mapc_count': len(chunk_mapcs),
    'question_count': len(questions),
    'gold_mapc_count': len(gold_mapcs),
    'missing_gold_mapc_count': len(missing_gold),
}])
display(validation)
assert not missing_gold, f'Missing gold MAPC: {missing_gold}'
print('Gold validation: PASS')

`chunk_count` là số chunk (một Điều dài có thể có nhiều chunk), `unique_mapc_count` là số Điều. `gold_mapc_count` là số Điều xuất hiện trong nhãn vàng; `missing_gold_mapc_count` phải bằng 0.


## 4. Tạo collection và chạy benchmark

Mỗi model có một collection Qdrant riêng `<prefix>_<model>` (prefix lấy từ manifest, ví dụ `legal_ld_v5_bge`). Collection chỉ được dùng lại (`index_action = reused`) khi có đúng tập `chunk_id` **và** đúng text; nếu không, benchmark tạo lại collection đó. `legal_ld_v5_bge` cũng là collection của chatbot, được tạo bằng `embed_and_index`.

```powershell
python -m labor_rag.evaluation.embedding_benchmark --benchmark lao_dong_v5 --testset v1
python -m labor_rag.evaluation.embedding_benchmark --benchmark lao_dong_v5 --testset v1 --model bge   # chỉ một model
```

Mỗi lần chạy ghi vào thư mục riêng `results/<dataset_id>/<testset>/<run_id>/`. Notebook tự lấy lần chạy mới nhất. Cấu hình mặc định: `top_k = 10`, warm-up 3 câu, mỗi câu chạy 3 lần, CI 95% bằng bootstrap 10.000 lần.


In [ ]:
RUN_BENCHMARK = False
if RUN_BENCHMARK:
    from labor_rag.evaluation.embedding_benchmark import run_benchmark
    benchmark_output = run_benchmark(benchmark=benchmark, top_k=10, warmup=3, repeats=3, rebuild=False)
    print('Benchmark finished.')
else:
    print('Đang dùng kết quả đã có trong results/<dataset_id>/<testset>/.')

## 5. Bảng metric tổng hợp

In [ ]:
if SUMMARY_PATH.exists():
    summary = pd.read_csv(SUMMARY_PATH)
    per_query = pd.DataFrame(read_jsonl(PER_QUERY_PATH)) if PER_QUERY_PATH.exists() else pd.DataFrame()
    collection_table = summary[['model', 'model_id', 'collection', 'embedding_dimension', 'indexed_points']].copy()
    collection_table.columns = ['Model', 'Hugging Face ID', 'Qdrant collection', 'Dimension', 'Số vector']
    display(collection_table)
    display(summary)
    assert set(summary['collection']) <= {spec['collection'] for spec in MODELS}
    assert set(summary['indexed_points']) == {len(chunks)}
    metrics_ci = pd.read_csv(METRICS_PATH)
    breakdown = pd.read_csv(BREAKDOWN_PATH) if BREAKDOWN_PATH.exists() else pd.DataFrame()
else:
    summary = pd.DataFrame()
    per_query = pd.DataFrame()
    metrics_ci = pd.DataFrame()
    breakdown = pd.DataFrame()
    print('Chưa có summary.csv. Hãy chạy CLI hoặc đặt RUN_BENCHMARK=True rồi chạy lại các cell.')

### Ý nghĩa các cột kết quả

Một Điều được tính là **đúng** khi có grade ≥ 2 (grade 1 chỉ cho ngữ cảnh). Định nghĩa theo NitiBench (EMNLP 2025).

| Nhóm | Cột | Ý nghĩa |
|---|---|---|
| Model | `model`, `model_id`, `collection`, `max_seq_length` | Model, collection Qdrant, số token tối đa model đọc (bkai/mpnet chỉ đọc phần đầu chunk). |
| Index | `embedding_dimension`, `indexed_points`, `index_action`, `index_time_seconds` | Số chiều, số vector, tạo lại hay dùng lại collection. |
| Truy hồi | `HitRate@1/5/10` | Tỷ lệ câu hỏi có **ít nhất một** Điều đúng trong top-k. |
| Truy hồi | `Recall@5/10` | Số Điều đúng tìm được / tổng số Điều đúng (micro, gộp mọi câu). |
| Truy hồi | `MultiHit@5/10` | Tỷ lệ câu hỏi có **đủ mọi** Điều đúng trong top-k; quan trọng với câu cần nhiều Điều. |
| Xếp hạng | `MRR@10` | 1 / vị trí của Điều đúng đầu tiên. |
| Xếp hạng | `nDCG@10` | Chất lượng thứ tự top-10, tính cả grade 1-3. |
| Tốc độ | `mean/median_latency_ms`, `mean_encode/search_latency_ms` | Encode câu hỏi và tìm kiếm Qdrant (gồm độ trễ mạng nếu dùng Qdrant Cloud). |

`metrics.csv` có khoảng tin cậy 95% của từng metric; hai model có CI chồng nhau nhiều thì chưa đủ cơ sở nói model nào tốt hơn. `breakdown.csv` tách metric theo đề mục, kiểu câu hỏi và tag.


In [ ]:
if not summary.empty:
    metric_columns = [
        'model', 'embedding_dimension', 'HitRate@1', 'HitRate@5', 'HitRate@10', 'Recall@10', 'MultiHit@10',
        'MRR@10', 'nDCG@10', 'mean_latency_ms', 'median_latency_ms',
        'index_time_seconds',
    ]
    display(summary[metric_columns].style.format({
        'HitRate@1': '{:.3f}', 'HitRate@5': '{:.3f}', 'HitRate@10': '{:.3f}', 'Recall@10': '{:.3f}', 'MultiHit@10': '{:.3f}',
        'MRR@10': '{:.3f}', 'nDCG@10': '{:.3f}',
        'mean_latency_ms': '{:.2f}', 'median_latency_ms': '{:.2f}',
        'index_time_seconds': '{:.2f}',
    }))
else:
    print('Chưa có dữ liệu metric để hiển thị.')

if not metrics_ci.empty:
    ci_table = metrics_ci.assign(
        ci95=metrics_ci.apply(lambda row: f"{row['value']:.3f} [{row['ci_low']:.3f}, {row['ci_high']:.3f}]", axis=1)
    ).pivot(index='metric', columns='model', values='ci95').reindex(METRICS)
    display(Markdown('**Giá trị và khoảng tin cậy 95% (bootstrap theo câu hỏi)**'))
    display(ci_table)


## 6. HitRate@k theo model


In [ ]:
if not summary.empty:
    fig, ax = plt.subplots(figsize=(10, 5))
    summary.set_index('model')[['HitRate@1', 'HitRate@5', 'HitRate@10']].plot(
        kind='bar', ax=ax, ylim=(0, 1.05), rot=0,
        color=['#4C78A8', '#F58518', '#54A24B'],
    )
    ax.set_title(f'HitRate@k trên {len(questions)} câu hỏi ({BENCHMARK}/{TESTSET})')
    ax.set_ylabel('Tỷ lệ câu hỏi có ít nhất một điều luật đúng')
    ax.set_xlabel('Embedding model')
    ax.grid(axis='y', alpha=0.3)
    ax.legend(title='Metric', loc='upper left', bbox_to_anchor=(1.02, 1.0))
    for container in ax.containers:
        ax.bar_label(container, fmt='%.2f', padding=2, fontsize=8)
    fig.tight_layout(rect=[0, 0, 0.78, 1])
    plt.show()

### Nhận xét HitRate

HitRate@k đo tỷ lệ câu hỏi có ít nhất một Điều luật đúng trong top-k. Nhận xét bên dưới được sinh tự động từ `summary.csv`; xem CI ở bảng trên trước khi kết luận model nào tốt hơn.


In [ ]:
if not summary.empty:
    for metric in ['HitRate@1', 'HitRate@5', 'HitRate@10']:
        best_value = summary[metric].max()
        worst_value = summary[metric].min()
        winners = ', '.join(summary.loc[summary[metric] == best_value, 'model'])
        print(f'{metric}: {winners} tốt nhất ({best_value:.3f}); khoảng cách cao nhất-thấp nhất = {best_value - worst_value:.3f}')
    best_top1 = summary.loc[summary['HitRate@1'].idxmax(), 'model']
    best_top10 = summary.loc[summary['HitRate@10'].idxmax(), 'model']
    print(f'\nDiễn giải: {best_top1} tìm đúng Điều luật ngay ở top-1 tốt nhất; {best_top10} có độ bao phủ top-10 tốt nhất.')
    print('Khi k tăng từ 1 lên 10, HitRate tăng vì hệ thống được phép xem nhiều ứng viên hơn.')

## 7. MRR và nDCG

In [ ]:
if not summary.empty:
    ax = summary.set_index('model')[['MRR@10', 'nDCG@10']].plot(
        kind='bar', figsize=(8, 5), ylim=(0, 1.05), rot=0,
        color=['#E45756', '#72B7B2'],
    )
    ax.set_title('Chất lượng xếp hạng trong top-10')
    ax.set_ylabel('Điểm')
    ax.set_xlabel('Embedding model')
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.show()

### Nhận xét MRR và nDCG

MRR cho biết Điều luật đúng đầu tiên thường đứng ở vị trí nào; nDCG xét thêm mức độ liên quan của nhiều kết quả. Điểm cao hơn nghĩa là thứ tự xếp hạng tốt hơn.

In [ ]:
if not summary.empty:
    for metric in ['MRR@10', 'nDCG@10']:
        winner_row = summary.loc[summary[metric].idxmax()]
        print(f'{metric}: {winner_row["model"]} tốt nhất ({winner_row[metric]:.3f}); chênh lệch với model thấp nhất = {winner_row[metric] - summary[metric].min():.3f}')
    print('Diễn giải: model có MRR cao thường đưa Điều luật đúng lên vị trí sớm hơn, giúp giảm số kết quả người dùng phải xem.')

## 8. Latency truy vấn

In [ ]:
if not per_query.empty:
    latency = per_query[['model', 'question_id', 'latency_ms', 'encode_latency_ms', 'search_latency_ms']].copy()
    display(latency.groupby('model')[['encode_latency_ms', 'search_latency_ms', 'latency_ms']].agg(['mean', 'median']).round(2))
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    latency.boxplot(column='latency_ms', by='model', ax=axes[0], grid=False)
    axes[0].set_title('Phân phối tổng latency')
    axes[0].set_xlabel('Embedding model')
    axes[0].set_ylabel('ms')
    summary.set_index('model')['median_latency_ms'].plot(kind='bar', ax=axes[1], rot=0, color='#B279A2')
    axes[1].set_title('Median latency')
    axes[1].set_xlabel('Embedding model')
    axes[1].set_ylabel('ms')
    axes[1].grid(axis='y', alpha=0.3)
    fig.suptitle('')
    plt.tight_layout()
    plt.show()
else:
    print('Chưa có per_query_results.jsonl để vẽ latency.')

### Nhận xét latency

Latency thấp hơn nghĩa là phản hồi nhanh hơn. Cần tách thời gian encode của model và thời gian Qdrant search để biết nút thắt nằm ở đâu.

In [ ]:
if not summary.empty:
    fastest = summary.loc[summary['median_latency_ms'].idxmin()]
    slowest = summary.loc[summary['median_latency_ms'].idxmax()]
    print(f'Median latency thấp nhất: {fastest["model"]} ({fastest["median_latency_ms"]:.2f} ms)')
    print(f'Median latency cao nhất: {slowest["model"]} ({slowest["median_latency_ms"]:.2f} ms)')
    if fastest['median_latency_ms'] > 0:
        print(f'{slowest["model"]} chậm khoảng {slowest["median_latency_ms"] / fastest["median_latency_ms"]:.1f} lần so với model nhanh nhất.')
    if (summary['index_action'] == 'reused').all():
        print('Lưu ý: các collection hiện được reuse nên index_time_seconds không dùng để so sánh thời gian xây dựng index trong lần chạy này.')

## 9. Heatmap hit top-1 theo từng câu hỏi

In [ ]:
if not per_query.empty:
    top1 = per_query.pivot(index='question_id', columns='model', values='hit_at_1').sort_index()
    fig, ax = plt.subplots(figsize=(8, 9))
    image = ax.imshow(top1.to_numpy(), aspect='auto', cmap='RdYlGn', vmin=0, vmax=1)
    ax.set_xticks(range(len(top1.columns)), top1.columns)
    ax.set_yticks(range(len(top1.index)), top1.index)
    ax.set_title('Top-1 hit theo câu hỏi')
    ax.set_xlabel('Embedding model')
    ax.set_ylabel('Question id')
    for row_index in range(top1.shape[0]):
        for col_index in range(top1.shape[1]):
            ax.text(col_index, row_index, int(top1.iloc[row_index, col_index]), ha='center', va='center')
    fig.colorbar(image, ax=ax, ticks=[0, 1], label='1 = hit, 0 = miss')
    plt.tight_layout()
    plt.show()

## 10. Ví dụ top-5 của cùng một câu hỏi

In [ ]:
if not per_query.empty:
    example_id = sorted(per_query['question_id'].unique())[0]
    example_question = per_query.loc[per_query['question_id'] == example_id, 'question'].iloc[0]
    display(Markdown(f'**{example_id}:** {example_question}'))
    for model_name, row in per_query[per_query['question_id'] == example_id].groupby('model', sort=False):
        results = row.iloc[0]['top_k_results'][:5]
        table = pd.DataFrame(results)
        display(Markdown(f'### {model_name}'))
        display(table[['rank', 'mapc', 'chunk_id', 'score', 'ten']])

## 11. Kết luận cuối cùng

Phần này tổng hợp kết quả từ `summary.csv`, không hard-code model thắng. Điểm cao hơn là tốt hơn với HitRate/Recall/MRR/nDCG; latency thấp hơn là tốt hơn. Kết luận chỉ nói về vector retrieval, chưa phải chất lượng câu trả lời cuối của LLM.

In [ ]:
if not summary.empty:
    chunks_count, mapc_count = len(chunks), len(chunk_mapcs)
    higher_is_better = ['HitRate@1', 'HitRate@5', 'HitRate@10', 'Recall@10', 'MultiHit@10', 'MRR@10', 'nDCG@10']

    def format_metric(metric, value):
        return f'{value:.1%}' if metric.startswith(('HitRate', 'Recall', 'MultiHit')) else f'{value:.3f}'

    def winner_label(metric):
        best_value = summary[metric].max()
        winners = summary.loc[summary[metric] == best_value, 'model']
        return ', '.join(f'`{model}`' for model in winners), best_value

    lines = [
        '### Kết quả chính',
        '',
        '| Tiêu chí | Model tốt nhất | Giá trị |',
        '|---|---|---:|',
    ]
    for metric in higher_is_better:
        winner_names, best_value = winner_label(metric)
        lines.append(f'| `{metric}` | {winner_names} | {format_metric(metric, best_value)} |')
    fastest_row = summary.loc[summary['median_latency_ms'].idxmin()]
    lines.append(f'| `Median latency` | `{fastest_row["model"]}` | {fastest_row["median_latency_ms"]:.2f} ms |')

    top1_row = summary.loc[summary['HitRate@1'].idxmax()]
    top10_row = summary.loc[summary['HitRate@10'].idxmax()]
    mrr_row = summary.loc[summary['MRR@10'].idxmax()]
    ndcg_row = summary.loc[summary['nDCG@10'].idxmax()]
    slowest_row = summary.loc[summary['median_latency_ms'].idxmax()]
    quality_winner = summary.loc[summary['MRR@10'].idxmax()]

    lines.extend([
        '',
        '### Diễn giải kết quả',
        f'- `{top1_row["model"]}` tốt nhất ở HitRate@1 ({top1_row["HitRate@1"]:.1%}), phù hợp khi cần đưa Điều luật đúng lên kết quả đầu tiên.',
        f'- `{top10_row["model"]}` tốt nhất ở HitRate@10 ({top10_row["HitRate@10"]:.1%}), phù hợp khi ưu tiên độ bao phủ trong top-10.',
        f'- `{mrr_row["model"]}` có MRR@10 cao nhất ({mrr_row["MRR@10"]:.3f}) và `{ndcg_row["model"]}` có nDCG@10 cao nhất ({ndcg_row["nDCG@10"]:.3f}), phản ánh chất lượng xếp hạng.',
        f'- `{fastest_row["model"]}` nhanh nhất với median latency {fastest_row["median_latency_ms"]:.2f} ms; `{slowest_row["model"]}` chậm nhất với {slowest_row["median_latency_ms"]:.2f} ms.',
    ])
    if quality_winner['model'] != slowest_row['model']:
        lines.append(f'- `{quality_winner["model"]}` là lựa chọn baseline cân bằng hiện tại vì đứng đầu MRR@10 nhưng không phải model chậm nhất.')
    if len(set(summary.loc[summary[higher_is_better].idxmax().values, 'model'])) > 1:
        lines.append('- Không có model thắng tuyệt đối: kết quả phụ thuộc vào việc ưu tiên vị trí đầu, độ bao phủ top-10 hay tốc độ.')

    lines.extend([
        '',
        '### Giới hạn của kết luận',
        f'- Benchmark {BENCHMARK}/{TESTSET}: {chunks_count} chunk ({mapc_count} Điều) và {len(questions)} câu hỏi vàng.',
        '- Chưa fine-tune model, chưa đánh giá Neo4j graph expansion và chưa đánh giá chất lượng câu trả lời cuối của LLM.',
        '- Latency phụ thuộc phần cứng, cache và độ trễ mạng tới Qdrant (Qdrant Cloud chậm hơn Docker local).',
    ])
    if (summary['index_action'] == 'reused').all():
        lines.append('- Các collection hiện được reuse nên index_time_seconds = 0; chưa dùng kết quả này để so sánh thời gian xây dựng index.')

    lines.extend([
        '',
        '### Hướng phát triển',
        '1. Mở rộng bộ câu hỏi và corpus pháp luật.',
        '2. Đánh giá riêng theo từng nhóm câu hỏi.',
        '3. Tạo benchmark hybrid Qdrant + Neo4j.',
        '4. Đánh giá citation và câu trả lời LLM trước khi chọn model triển khai chính.',
    ])
    display(Markdown('\n'.join(lines)))
else:
    print('Chưa có kết quả để kết luận.')